# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Final Sample Construction

---
## Foreword

In this notebook, my goal is to build the final main sample for the models developed in my thesis.


## 1. Libraries & Data

I first load the relevant libraries and data.

In [1]:
# libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
# data
main = pd.read_csv("../../../data/processed/main_with_svi.csv")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_22132/1285260102.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../../data/processed/main_with_svi.csv")


## 2. Missing Values & EDA

In this section, the goal is to identify the missing values that will be an issue for my analysis. The main constraint I will face is the serious missingness in Google Trends search volume index (SVI).

In [33]:
# parse dates
main['date'] = pd.to_datetime(main['date'])

# rename TSSVI to svi
main.rename(columns={'TSSVI': 'svi'}, inplace=True)

### 2.1. Missing Google Trends Data

I study the missingness of Google Trends data.

In [34]:
# number of unique tickers in the google svi dataset
svi = pd.read_csv("../../../data/interim/google_trends/google_trends_svi.csv")

# unique tickers in svi dataset
unique_tickers_svi = svi['ticker'].unique()
print(f"Number of unique tickers in SVI dataset: {len(unique_tickers_svi)}")

# unique tickers in main dataset
unique_tickers_main = main['ticker'].unique()
print(f"Number of unique tickers in main dataset: {len(unique_tickers_main)}")

# intersection of tickers in main and svi datasets
intersection_tickers = set(unique_tickers_main).intersection(set(unique_tickers_svi))
print(f"Number of tickers in intersection: {len(intersection_tickers)}")

Number of unique tickers in SVI dataset: 3034
Number of unique tickers in main dataset: 8593
Number of tickers in intersection: 1979


This means that in the attention sample I will have 1723 tickers.

#### herding events

I also study the influence on the number of herding events.

In [35]:
# sort values by ticker and date
rh_herd = main[main["ticker"].isin(intersection_tickers)].copy()  # create a copy to avoid SettingWithCopyWarning
rh_herd = rh_herd.sort_values(["ticker", "date"]).copy()

# userratio = users_close(t) / users_close(t-1)
rh_herd["users_close_lag1"] = rh_herd.groupby("ticker")["users_close"].shift(1)
rh_herd["userratio"] = rh_herd["users_close"] / rh_herd["users_close_lag1"]

# flag np.inf and -np.inf values in userratio as NaN (can happen when users_close_lag1 is zero)
rh_herd["userratio"].replace([np.inf, -np.inf], np.nan, inplace=True)

# shortlist the possible herding events based on the criteria of Barbet et al. (2022)
rh_herd["shortlist_herd"] = (
    (rh_herd["userratio"] > 1)
    & (rh_herd["users_close_lag1"] >= 100)
    & rh_herd["userratio"].notna()
)

# Identify the top 0.5% of shortlisted stocks based on userratio for each day
# Only shortlisted stocks can be flagged as herding events.
rh_herd["rh_herd"] = False
rh_herd.loc[rh_herd["shortlist_herd"], "rh_herd"] = (
    rh_herd.loc[rh_herd["shortlist_herd"]]
    .groupby("date")["userratio"]
    .transform(lambda x: x >= x.quantile(0.995))
)

# Report on the number of herding events identified, number of
# unique tickers involved
num_herding_events = rh_herd["rh_herd"].sum()
num_unique_tickers = rh_herd.loc[rh_herd["rh_herd"], "ticker"].nunique()
print(f"Number of herding events identified: {num_herding_events}")
print(f"Number of unique tickers involved: {num_unique_tickers}")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_22132/4283351143.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  rh_herd["userratio"].replace([np.inf, -np.inf], np.nan, inplace=True)


Number of herding events identified: 2349
Number of unique tickers involved: 926


This means that I still have a meaningful number of herding events.

#### summary statistics

I compute the summary statistics to compare it to my previous sample.

In [36]:
# restrict main to the tickers in the intersection of main and svi datasets
main = main[main["ticker"].isin(intersection_tickers)].copy()

In [37]:
# users_close_l1: lagged users_close by 1 day
main["users_close_l1"] = main.groupby("ticker")["users_close"].shift(1)

# userchg: change in users from day t-1 to day t
main["userchg"] = main["users_close"] - main["users_close_l1"]

# userratio: users_close(t) / users_close(t-1)
main["userratio"] = main["users_close"] / main["users_close_l1"]

# replace inf and -inf in userratio with NaN
main["userratio"] = main["userratio"].replace([np.inf, -np.inf], np.nan)

# make sure that I take the absolute value of the price
main["prc"] = main["prc"].abs()

# size is defined as prc * shrout
main["size"] = main["prc"] * main["shrout"] / 1e6  # in millions

# build 'daily_buys'
main["daily_buys"] = main["buy_num_trades_retail"]

# build 'daily_sells'
main["daily_sells"] = main["sell_num_trades_retail"]

# build 'net_buys'
main["net_buys"] = main["daily_buys"] - main["daily_sells"]

# build 'taq_retimb'
main['taq_retimb'] = main['net_buys'] / (main['daily_buys'] + main['daily_sells'])

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
main["has_users"] = (main["users_close"] > 0).astype(int)

In [38]:
# summary stats Panel A
summary_stats_panel_a = main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb", "svi"]].describe().T
print(summary_stats_panel_a)

                 count         mean           std           min        25%  \
users_close  1519954.0  3701.030219  23563.294690      0.000000  90.000000   
users_last   1565355.0  3707.178004  23628.000255      0.000000  90.000000   
userchg      1477712.0    11.097429    259.629021 -17015.000000  -1.000000   
userratio    1454700.0     1.003655      0.231604      0.000000   0.998657   
prc          1085766.0    66.946998    151.602584      0.085000  16.240000   
size         1085766.0    14.518053     55.368101      0.001172   0.674598   
ret          1085766.0     0.000376      0.036546     -1.000000  -0.011344   
daily_buys   1080275.0   334.782329   1490.298060      0.000000  23.000000   
daily_sells  1080275.0   307.030789   1208.494935      0.000000  23.000000   
net_buys     1080275.0    27.751540    445.943083 -10131.000000 -12.000000   
taq_retimb   1080275.0     0.001530      0.235921     -1.000000  -0.100000   
svi          1442176.0    17.705679     29.350945      0.000000 

## 3. Building the Final Sample

I now build the final samples for my analysis.

In [39]:
# read data
df_final = pd.read_csv("../../../data/processed/main_with_svi.csv")

# parse dates
df_final['date'] = pd.to_datetime(df_final['date'])

# rename TSSVI to svi
df_final.rename(columns={'TSSVI': 'svi'}, inplace=True)

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_22132/571122779.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv("../../../data/processed/main_with_svi.csv")


In [40]:
# new main
new_main = df_final.copy()
new_main["prc"] = new_main["prc"].abs()

In [41]:
# Missing values
num_missing_prc = new_main["prc"].isna().sum()
num_missing_ret = new_main["ret"].isna().sum()
num_missing_shrout = new_main["shrout"].isna().sum()
print(f"Number of missing values in 'prc': {num_missing_prc}")
print(f"Number of missing values in 'ret': {num_missing_ret}")
print(f"Number of missing values in 'shrout': {num_missing_shrout}")

num_missing_close_price = new_main["close_price"].isna().sum()
num_missing_open_price = new_main["open_price"].isna().sum()
print(f"Number of missing values in 'close_price': {num_missing_close_price}")
print(f"Number of missing values in 'open_price': {num_missing_open_price}")

# Tickers with no CRSP price at all
tickers_no_price_data = new_main["prc"].isna().groupby(new_main["ticker"]).all()
num_tickers_no_price_data = tickers_no_price_data.sum()
print(f"Number of tickers with no 'prc' data at all: {num_tickers_no_price_data}")

# Tickers with no TAQ close price at all
tickers_no_close_price_data = new_main["close_price"].isna().groupby(new_main["ticker"]).all()
num_tickers_no_close_price_data = tickers_no_close_price_data.sum()
print(f"Number of tickers with no 'close_price' data at all: {num_tickers_no_close_price_data}")

# Tickers with neither CRSP nor TAQ close price data at all
tickers_no_price_and_close_price_data = tickers_no_price_data & tickers_no_close_price_data
num_tickers_no_price_and_close_price_data = tickers_no_price_and_close_price_data.sum()
print(f"Number of tickers with neither 'prc' nor 'close_price' data at all: {num_tickers_no_price_and_close_price_data}")

# Drop those tickers
tickers_to_drop = tickers_no_price_and_close_price_data.index[tickers_no_price_and_close_price_data]
new_main = new_main[~new_main["ticker"].isin(tickers_to_drop)].copy()

print(f"Number of tickers dropped: {len(tickers_to_drop)}")

Number of missing values in 'prc': 2184440
Number of missing values in 'ret': 2184440
Number of missing values in 'shrout': 2184440
Number of missing values in 'close_price': 2133976
Number of missing values in 'open_price': 2130308
Number of tickers with no 'prc' data at all: 518
Number of tickers with no 'close_price' data at all: 321
Number of tickers with neither 'prc' nor 'close_price' data at all: 321
Number of tickers dropped: 321


Once this is done, I recompute the stats above.

In [42]:
# RETURNS AND PRICE IMPUTATIONS WITH TAQ AND CRSP
# I look up the number of missing variables for 'prc' and 'ret' in the new_main dataset
num_missing_prc = new_main["prc"].isna().sum()
num_missing_ret = new_main["ret"].isna().sum()
num_missing_shrout = new_main["shrout"].isna().sum()
print(f"Number of missing values in 'prc': {num_missing_prc}")
print(f"Number of missing values in 'ret': {num_missing_ret}")
print(f"Number of missing values in 'shrout': {num_missing_shrout}")

# I look up the number of missing variable for 'close_price' and 'open_price' in the new_main dataset
num_missing_close_price = new_main["close_price"].isna().sum()
num_missing_open_price = new_main["open_price"].isna().sum()
print(f"Number of missing values in 'close_price': {num_missing_close_price}")
print(f"Number of missing values in 'open_price': {num_missing_open_price}")

# I look up the number of tickers with no price data at all (i.e., all values for 'prc' are missing) in the new_main dataset
tickers_no_price_data = new_main.groupby("ticker")["prc"].apply(lambda x: x.isna().all())
num_tickers_no_price_data = tickers_no_price_data.sum()
print(f"Number of tickers with no price data at all: {num_tickers_no_price_data}")

# I look up the number of tickers with no close_price data at all (i.e., all values for 'close_price' are missing) in the new_main dataset
tickers_no_close_price_data = new_main.groupby("ticker")["close_price"].apply(lambda x: x.isna().all())
num_tickers_no_close_price_data = tickers_no_close_price_data.sum()
print(f"Number of tickers with no close_price data at all: {num_tickers_no_close_price_data}")

# Intersection of tickers with no price data and tickers with no close_price data
tickers_no_price_and_close_price_data = tickers_no_price_data & tickers_no_close_price_data
num_tickers_no_price_and_close_price_data = tickers_no_price_and_close_price_data.sum()
print(f"Number of tickers with no price data and no close_price data: {num_tickers_no_price_and_close_price_data}")

Number of missing values in 'prc': 1982635
Number of missing values in 'ret': 1982635
Number of missing values in 'shrout': 1982635
Number of missing values in 'close_price': 1932171
Number of missing values in 'open_price': 1928503
Number of tickers with no price data at all: 197
Number of tickers with no close_price data at all: 0
Number of tickers with no price data and no close_price data: 0


I now chek if I can impute the missing values with TAQ and CRSP data. 

In [43]:
# when we have value both for 'prc' and 'close' price, for a given ticker and date, I compute their absolute difference
new_main["price_diff"] = (new_main["prc"] - new_main["close_price"]).abs()

# I look up the summary statistics for the price difference between 'prc' and 'close_price'
price_diff_summary_stats = new_main["price_diff"].describe()
print("Summary statistics for price difference between 'prc' and 'close_price':")
print(price_diff_summary_stats)

Summary statistics for price difference between 'prc' and 'close_price':
count    3.818395e+06
mean     1.247864e-02
std      2.732038e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.139999e+01
Name: price_diff, dtype: float64


I now observe where the price diff is greater than 1

In [44]:
# observe where the price difference is greater than 1 dollar
price_diff_greater_than_1 = new_main[new_main["price_diff"] > 1][["ticker", "date", "prc", "close_price", "price_diff"]]
print("Observations where price difference between 'prc' and 'close_price' is greater than 1 dollar:")
print(price_diff_greater_than_1)

# percentage of observations where price difference is greater than 1 dollar
percentage_price_diff_greater_than_1 = (price_diff_greater_than_1.shape[0] / new_main.shape[0]) * 100
print(f"Percentage of observations where price difference between 'prc' and 'close_price' is greater than 1 dollar: {percentage_price_diff_greater_than_1:.2f}%")

Observations where price difference between 'prc' and 'close_price' is greater than 1 dollar:
        ticker       date      prc  close_price  price_diff
14340     GFED 2018-06-25  23.9400        22.72      1.2200
14434     GFED 2018-09-27  23.7700        25.35      1.5800
14440     GFED 2018-10-03  25.5100        24.16      1.3500
14462     GFED 2018-10-25  24.7000        23.38      1.3200
14476     GFED 2018-11-08  23.8000        22.50      1.3000
...        ...        ...      ...          ...         ...
5960878   SNDE 2020-01-07  19.4500        18.30      1.1500
5969576   HWBK 2018-10-02  23.9053        22.52      1.3853
5985272   ICCH 2020-03-16  11.5000        10.41      1.0900
5985274   ICCH 2020-03-18  10.4200         9.41      1.0100
5985393   ICCH 2020-07-15  11.9750        13.86      1.8850

[3643 rows x 5 columns]
Percentage of observations where price difference between 'prc' and 'close_price' is greater than 1 dollar: 0.06%


This is acceptable for the imputation.

In [45]:
# if prc is missing but close_price is not missing, I impute prc with close_price
new_main.loc[new_main["prc"].isna() & new_main["close_price"].notna(), "prc"] = new_main["close_price"]

# if close_price is missing but prc is not missing, I impute close_price with prc
new_main.loc[new_main["close_price"].isna() & new_main["prc"].notna(), "close_price"] = new_main["prc"]

# prc_lag1: lagged prc by 1 day
new_main["prc_lag1"] = new_main.groupby("ticker")["prc"].shift(1)

# 'ret' imputation: if ret is missing but prc and prc_lag1 are not missing, I impute ret with (prc - prc_lag1) / prc_lag1
new_main.loc[new_main["ret"].isna() & new_main["prc"].notna() & new_main["prc_lag1"].notna(), "ret"] = (
    (new_main["prc"] - new_main["prc_lag1"]) / new_main["prc_lag1"]
)

# I look up the number of missing variables for 'prc' and 'ret' in the new_main dataset after the imputations
num_missing_prc_after = new_main["prc"].isna().sum()
num_missing_ret_after = new_main["ret"].isna().sum()
print(f"Number of missing values in 'prc' after imputation: {num_missing_prc_after}")
print(f"Number of missing values in 'ret' after imputation: {num_missing_ret_after}")

# I look up the number of tickers with no price data at all (i.e., all values for 'prc' are missing) in the new_main dataset after the imputations
tickers_no_price_data_after = new_main.groupby("ticker")["prc"].apply(lambda x: x.isna().all())
num_tickers_no_price_data_after = tickers_no_price_data_after.sum()
print(f"Number of tickers with no price data at all after imputation: {num_tickers_no_price_data_after}")

Number of missing values in 'prc' after imputation: 1906049
Number of missing values in 'ret' after imputation: 1924594
Number of tickers with no price data at all after imputation: 0


Now, I observe a row for which we have missing `prc` data, specifically, I observe the row before and after.

In [46]:
# make sure data are ordered properly
new_main = new_main.sort_values(['ticker', 'date']).reset_index(drop=True)

# indices where prc is missing
missing_idx = new_main.index[new_main['prc'].isna()]

# show the two rows before, the missing row, and two rows after
for i in missing_idx[100:105]:
    print(new_main.loc[max(0, i-2):min(len(new_main)-1, i+2)][['date', 'ticker', 'prc', 'ret']])
    print("-" * 80)

          date ticker    prc       ret
317 2019-03-15      A  81.10  0.006079
318 2019-03-16      A    NaN       NaN
319 2019-03-17      A    NaN       NaN
320 2019-03-18      A  80.97 -0.001603
321 2019-03-19      A  81.20  0.002841
--------------------------------------------------------------------------------
          date ticker    prc       ret
323 2019-03-21      A  82.00  0.013347
324 2019-03-22      A  78.99 -0.036707
325 2019-03-23      A    NaN       NaN
326 2019-03-24      A    NaN       NaN
327 2019-03-25      A  79.52  0.006710
--------------------------------------------------------------------------------
          date ticker    prc       ret
324 2019-03-22      A  78.99 -0.036707
325 2019-03-23      A    NaN       NaN
326 2019-03-24      A    NaN       NaN
327 2019-03-25      A  79.52  0.006710
328 2019-03-26      A  80.65  0.014210
--------------------------------------------------------------------------------
          date ticker    prc       ret
330 2019-03-28  

The current issue is likely linked to the fact that the data set contains weekends, i.e. no trading occur and no data are available. To remedy to this issue, I create a binary variable called `trading_day`$= 1$ if it is a valid trading day, and `trading_day`$= 0$ if not.

In [47]:
new_main['date'] = pd.to_datetime(new_main['date'])

# 1 if Monday-Friday, 0 if Saturday-Sunday
new_main['trading_day'] = (new_main['date'].dt.dayofweek < 5).astype(int)

# I now count the number of missing values in 'prc' and 'ret' for trading days only (i.e., I exclude non-trading days) in the new_main dataset
num_missing_prc_trading_days = new_main.loc[new_main['trading_day'] == 1, 'prc'].isna().sum()
num_missing_ret_trading_days = new_main.loc[new_main['trading_day'] == 1, 'ret'].isna().sum()
print(f"Number of missing values in 'prc' for trading days: {num_missing_prc_trading_days}")
print(f"Number of missing values in 'ret' for trading days: {num_missing_ret_trading_days}")

# percentage of missing values in 'prc' and 'ret' for trading days only in the new_main dataset
total_trading_days = new_main['trading_day'].sum()
perc_missing_prc_trading_days = num_missing_prc_trading_days / total_trading_days * 100
perc_missing_ret_trading_days = num_missing_ret_trading_days / total_trading_days * 100
print(f"Percentage of missing values in 'prc' for trading days: {perc_missing_prc_trading_days:.2f}%")
print(f"Percentage of missing values in 'ret' for trading days: {perc_missing_ret_trading_days:.2f}%")

Number of missing values in 'prc' for trading days: 245389
Number of missing values in 'ret' for trading days: 263934
Percentage of missing values in 'prc' for trading days: 5.89%
Percentage of missing values in 'ret' for trading days: 6.33%


Now I define a new variable called `strict_trading_day` that takes the value $1$ if it is a weekday and we have data for `prc`.

In [48]:
new_main['strict_trading_day'] = (
    (new_main['trading_day'] == 1) & new_main['prc'].notna()
).astype(int)

I perform further check to compute the percentage of strict trading day in my data set.

In [49]:
# percentage of strict trading days in the new_main dataset
total_strict_trading_days = new_main['strict_trading_day'].sum()
perc_strict_trading_days = total_strict_trading_days / len(new_main) * 100
print(f"Percentage of strict trading days in the new_main dataset: {perc_strict_trading_days:.2f}%")

# percentage of trading days in the new_main dataset
total_trading_days = new_main['trading_day'].sum()
perc_trading_days = total_trading_days / len(new_main) * 100
print(f"Percentage of trading days in the new_main dataset: {perc_trading_days:.2f}%")

Percentage of strict trading days in the new_main dataset: 67.29%
Percentage of trading days in the new_main dataset: 71.50%


I now look at the number of missing `shrout` during strict trading days.

In [50]:
# missing values in 'shrout' for strict trading days = 1
num_missing_shrout_strict_trading_days = new_main.loc[new_main['strict_trading_day'] == 1, 'shrout'].isna().sum()
print(f"Number of missing values in 'shrout' for strict trading days: {num_missing_shrout_strict_trading_days}")

# percentage of missing values in 'shrout' for strict trading days
total_strict_trading_days = new_main['strict_trading_day'].sum()
perc_missing_shrout_strict_trading_days = num_missing_shrout_strict_trading_days / total_strict_trading_days * 100
print(f"Percentage of missing values in 'shrout' for strict trading days: {perc_missing_shrout_strict_trading_days:.2f}%")


Number of missing values in 'shrout' for strict trading days: 76586
Percentage of missing values in 'shrout' for strict trading days: 1.95%


I look up the number of tickers for which we have no 'shrout' values at all.

In [51]:
# I look up the number of tickers for which we have no 'shrout' values at all.
num_tickers_no_shrout = new_main.groupby('ticker')['shrout'].apply(lambda x: x.isna().all()).sum()
print(f"Number of tickers with no 'shrout' values at all: {num_tickers_no_shrout}")

# I create a variable 'shrout_missing' that takes the value 1 if 'shrout' is never reported for that ticker and 0 otherwise
new_main['shrout_missing'] = new_main.groupby('ticker')['shrout'].transform(lambda x: int(x.isna().all()))

# number of rows in new_main where shrout is missing and strict_trading_day is 1
num_rows_shrout_missing_strict_trading_day = new_main.loc[(new_main['shrout_missing'] == 1) & (new_main['strict_trading_day'] == 1)].shape[0]
print(f"Number of rows where shrout is missing and strict_trading_day is 1: {num_rows_shrout_missing_strict_trading_day}")

Number of tickers with no 'shrout' values at all: 197
Number of rows where shrout is missing and strict_trading_day is 1: 75666


I now observe a ticker with no shrout values.

In [52]:
# a ticker for which we have no 'shrout' values at all
tickers_no_shrout = new_main.groupby('ticker')['shrout'].apply(lambda x: x.isna().all())

# a few examples of tickers with no 'shrout' values at all
tickers_no_shrout_examples = tickers_no_shrout[tickers_no_shrout].index[:5]
print(f"Examples of tickers with no 'shrout' values at all: {tickers_no_shrout_examples.tolist()}")

Examples of tickers with no 'shrout' values at all: ['AMJ', 'AMJL', 'AMNA', 'AMND', 'AMU']


The presence of valid price and return data alongside systematically missing shares-outstanding values is consistent with the inclusion of non-common-equity instruments in the Robintrack universe. For example, the ticker JPMorgan Alerian MLP Index ETN is an exchange-traded note rather than an ordinary common share. Such instruments can have observable market prices and returns, while shares outstanding are either not economically comparable to those of common equity or unavailable in CRSP. Such concerns are NOT addressed in Barber et al. (2022).

In [53]:
# unique tickers in Barber et al. (2022) dataset
barber1 = pd.read_stata("../../../data/replication/main_pseudo.dta")
barber2 = pd.read_stata("../../../data/replication/topmover_pseudo.dta")

# unique tickers in Barber et al. (2022) datasets
unique_tickers_barber1 = barber1["ticker"].unique()
unique_tickers_barber2 = barber2["ticker"].unique()
print(f"Number of unique tickers in Barber et al. (2022) pseudo_main dataset: {len(unique_tickers_barber1)}")
print(f"Number of unique tickers in Barber et al. (2022) pseudo_topmover dataset: {len(unique_tickers_barber2)}")


Number of unique tickers in Barber et al. (2022) pseudo_main dataset: 8447
Number of unique tickers in Barber et al. (2022) pseudo_topmover dataset: 8295


In [54]:
# look if the tickers with no 'shrout' values at all in our new_main dataset are also present in the Barber et al. (2022) datasets
tickers_no_shrout_in_barber1 = set(tickers_no_shrout[tickers_no_shrout].index).intersection(set(unique_tickers_barber1))
tickers_no_shrout_in_barber2 = set(tickers_no_shrout[tickers_no_shrout].index).intersection(set(unique_tickers_barber2))
print(f"Number of tickers with no 'shrout' values at all in new_main that are also present in Barber et al. (2022) pseudo_main dataset: {len(tickers_no_shrout_in_barber1)}")
print(f"Number of tickers with no 'shrout' values at all in new_main that are also present in Barber et al. (2022) pseudo_topmover dataset: {len(tickers_no_shrout_in_barber2)}")

Number of tickers with no 'shrout' values at all in new_main that are also present in Barber et al. (2022) pseudo_main dataset: 193
Number of tickers with no 'shrout' values at all in new_main that are also present in Barber et al. (2022) pseudo_topmover dataset: 187


In [55]:
barber1.loc[barber1["ticker"] == "ACB"][["date", "users_close"]].tail()

,date,users_close
40846,20190513.0,545.0
45142,20190618.0,118.0
49913,20190726.0,488.0
59444,20191009.0,113.0
95340,20200713.0,2080.0


So we indeed see that Barber et al. (2022) were not attentive to this issue.

In [56]:
# number of unique tickers in my dataset
unique_tickers_new_main = new_main["ticker"].unique()
print(f"Number of unique tickers in my dataset: {len(unique_tickers_new_main)}")

# intersection of unique tickers in my dataset and Barber et al. (2022) datasets
intersection_barber1 = set(unique_tickers_new_main).intersection(set(unique_tickers_barber1))
intersection_barber2 = set(unique_tickers_new_main).intersection(set(unique_tickers_barber2))
print(f"Number of unique tickers in intersection with Barber et al. (2022) pseudo_main dataset: {len(intersection_barber1)}")
print(f"Number of unique tickers in intersection with Barber et al. (2022) pseudo_topmover dataset: {len(intersection_barber2)}")

Number of unique tickers in my dataset: 8272
Number of unique tickers in intersection with Barber et al. (2022) pseudo_main dataset: 8129
Number of unique tickers in intersection with Barber et al. (2022) pseudo_topmover dataset: 7985


Now to finalize my data cleaning, I will keep only valid trading days, as denoted by the variable `strict_trading_days`.

In [57]:
# drop all observations which have value 0 for 'strict_trading_day'
new_main = new_main[new_main['strict_trading_day'] == 1].copy()

# build relevant variables for new_main
new_main["users_close_l1"] = new_main.groupby("ticker")["users_close"].shift(1)

# userchg: change in users from day t-1 to day t
new_main["userchg"] = new_main["users_close"] - new_main["users_close_l1"]

# intraday user change: change in users from open to close
new_main["userchg_intraday"] = new_main["users_close"] - new_main["users_start"]

# userratio: users_close(t) / users_close(t-1)
new_main["userratio"] = new_main["users_close"] / new_main["users_close_l1"]

# replace inf and -inf in userratio with NaN
new_main["userratio"] = new_main["userratio"].replace([np.inf, -np.inf], np.nan)

# build 'daily_buys'
new_main["daily_buys"] = new_main["buy_num_trades_retail"]

# build 'daily_sells'
new_main["daily_sells"] = new_main["sell_num_trades_retail"]

# build 'net_buys'
new_main["net_buys"] = new_main["daily_buys"] - new_main["daily_sells"]

# build 'taq_retimb'
new_main['taq_retimb'] = new_main['net_buys'] / (new_main['daily_buys'] + new_main['daily_sells'])

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
new_main["has_users"] = (new_main["users_close"] > 0).astype(int)

# build abnormal SVI
def make_ab_svi(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

new_main["asvi"] = new_main.groupby("ticker")["svi"].transform(make_ab_svi)

Once that this is done, I further save two datasets. The *baseline trading sample* dataset, and the *attention sample* where I have tickers with `svi` observations. Importantly, I keep observations only when I have the variable `users_last` observed.

In [58]:
# percentage of the rows with a non-missing value for 'users_last' in the new_main dataset
percentage_non_missing_users_last = new_main['users_last'].notna().mean() * 100
print(f"Percentage of rows with non-missing 'users_last': {percentage_non_missing_users_last:.2f}%")

# users_close
percentage_non_missing_users_close = new_main['users_close'].notna().mean() * 100
print(f"Percentage of rows with non-missing 'users_close': {percentage_non_missing_users_close:.2f}%")

# users_start
percentage_non_missing_users_start = new_main['users_start'].notna().mean() * 100
print(f"Percentage of rows with non-missing 'users_start': {percentage_non_missing_users_start:.2f}%")

# I create a binary variable valid_rh that takes the value 1 if at least one of 'users_last' or 'users_close' are
# non-missing, and 0 otherwise
new_main['valid_rh'] = ((new_main['users_last'].notna()) | (new_main['users_close'].notna())).astype(int)

# percentage of rows with valid_rh = 1 in the new_main dataset
percentage_valid_rh = new_main['valid_rh'].mean() * 100
print(f"Percentage of rows with valid_rh = 1: {percentage_valid_rh:.2f}%")

Percentage of rows with non-missing 'users_last': 97.97%
Percentage of rows with non-missing 'users_close': 95.79%
Percentage of rows with non-missing 'users_start': 97.63%
Percentage of rows with valid_rh = 1: 97.97%


In [59]:
# filter the new_main dataset to keep only the rows with valid_rh = 1
new_main = new_main[new_main['valid_rh'] == 1].copy()

# save baseline_trading_sample
new_main.to_csv("../../../data/processed/baseline_trading_sample.csv", index=False)

# attention sample
##################

# list of tickers for which we have at least one non-missing 'svi' value
tickers_with_svi = new_main.groupby('ticker')['svi'].apply(lambda x: x.notna().any())
tickers_with_svi = tickers_with_svi[tickers_with_svi].index
print(f"Number of tickers with at least one non-missing 'svi' value: {len(tickers_with_svi)}")

# filter new_main to keep only the tickers with at least one non-missing 'svi' value
attention_sample = new_main[new_main['ticker'].isin(tickers_with_svi)].copy()

# save attention_sample
attention_sample.to_csv("../../../data/processed/attention_sample.csv", index=False)

Number of tickers with at least one non-missing 'svi' value: 1979
